# Fairlearn

This notebook walks through deckard's Fairlearn integration for both sklearn and PyTorch workflows, and includes attribute-inference attacks to illustrate privacy risk.

## Navigation

1. Environment and imports
2. Fairlearn-native dataset loading (Adult)
3. FairlearnDataConfig with preprocessing defense
4. sklearn fairness defense (`ExponentiatedGradient`)
5. Attribute-inference attack on sensitive feature
6. PyTorch fairness defense (`AdversarialFairnessClassifier`)
7. Yellowbrick plotter (sklearn fairness experiment)
8. Seaborn summary plots

The dataset source in this notebook is `fairlearn.datasets`, similar to how lifelines-native datasets are used in the survival notebook.

## 1. Optional Dependencies

This notebook focuses on fairness metrics and group-aware evaluation.

Recommended extras:

- `docs`
- `fairlearn`

Install from repository root:

```bash
pip install -e '.[fairlearn,seaborn,yellowbrick,torch]'
```

In [1]:
# All imports moved to the top for clarity and reproducibility
from pathlib import Path
from deckard.plugins.fairlearn.data import FairlearnDataConfig
from deckard.plugins.fairlearn.model import FairlearnModelConfig, FairlearnPytorchModelConfig
from deckard.frameworks.pytorch.model import TinyNet
from deckard.frameworks.pytorch.experiment import TorchExperimentConfig
from deckard.plugins.seaborn.plot import SeabornPlotConfig, SeabornPlotConfigList
from deckard.plugins.yellowbrick.plot import YellowbrickPlotConfig

NOTEBOOK_ROOT = Path(".")
ARTIFACT_DIR = NOTEBOOK_ROOT / "build" / "fairlearn"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Canonical output file paths for each step
data_file = ARTIFACT_DIR / "fairlearn_data.pkl"
fair_data_file = ARTIFACT_DIR / "fairlearn_fair_data.pkl"
sklearn_model_file = ARTIFACT_DIR / "fairlearn_sklearn_model.pkl"
torch_model_file = ARTIFACT_DIR / "fairlearn_torch_model.pt"
sklearn_score_file = ARTIFACT_DIR / "fairlearn_sklearn_scores.csv"
torch_score_file = ARTIFACT_DIR / "fairlearn_torch_scores.csv"
attack_file = ARTIFACT_DIR / "fairlearn_attack_results.pkl"
plot_file = ARTIFACT_DIR / "fairlearn_attack_plot.png"

print(f"Artifacts will be written under: {ARTIFACT_DIR}")

/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Artifacts will be written under: build/fairlearn


## 2. Adult Dataset Loading

This section uses deckard's built-in `adult` loader, which now performs the Adult target encoding, sensitive-feature encoding, and categorical expansion centrally in `DataConfig`.

In [2]:
adult_data = FairlearnDataConfig(
    dataset_name="adult",
    train_size=0.8,
    test_size=0.2,
    random_state=42,
    classifier=True,
    sensitive_columns="sex",
)
data_scores = adult_data(data_file=data_file.as_posix())
for k,v in data_scores.items():
    print(f"{k}:{v}")

data_load_time:0.071577
data_sample_time:0.006057999999999897
pipeline_fit_time:0.0
pipeline_fit_n:39073
pipeline_transform_time:0.0
pipeline_transform_n:9769
num_classes:2
class_count_min:11687
class_count_max:37155
class_imbalance_ratio:3.179173440574998
mutual_information_mean:0.00842203267129628
mutual_information_max:0.1113041974251816


## 3. FairlearnDataConfig with Preprocessing Defense

`FairlearnDataConfig` can inject a Fairlearn preprocessing defense (`CorrelationRemover`) while preserving sensitive-feature caches for fairness and attack scoring.

In [3]:
from deckard.plugins.fairlearn.data import FairlearnDataConfig

fair_data = FairlearnDataConfig(
    dataset_name="adult",
    classifier=True,
    stratify=False,
    train_size=0.8,
    test_size=0.2,
    random_state=42,
    sensitive_columns=["sex"],
    fairness_defense={
        "step_name": "fairness_correlation_remover",
        "name": "fairlearn.preprocessing.CorrelationRemover",
        "alpha": 1.0,
    },
)

fair_data_scores = fair_data(data_file=fair_data_file.as_posix(), score_file=sklearn_score_file.as_posix())

print("Train shape:", fair_data.X_train.shape)
print("Test shape:", fair_data.X_test.shape)
print("Sensitive train sample:", fair_data._sensitive_train.iloc[:5].tolist())
for k,v in fair_data_scores.items():
    print(f"{k}:{v}")

Train shape: (39073, 103)
Test shape: (9769, 103)
Sensitive train sample: ['0', '0', '0', '1', '0']
data_load_time:0.07842399999999827
data_sample_time:0.001020000000000465
pipeline_fit_time:0.036129999999999995
pipeline_fit_n:39073
pipeline_transform_time:0.0054149999999992815
pipeline_transform_n:9769
num_classes:2
class_count_min:11687
class_count_max:37155
class_imbalance_ratio:3.179173440574998
mutual_information_mean:0.00842203267129628
mutual_information_max:0.1113041974251816
accuracy:0.8127
precision:0.7975
recall:0.8127
f1:0.7979
roc_auc:0.7821
log_loss:0.4387
demographic_parity_difference:0.0311
equalized_odds_difference:0.2045
group_mean_prediction_difference:0.0311
0_accuracy:0.775
0_precision:0.7676
0_recall:0.775
0_f1:0.752
0_roc_auc:0.6678
0_log_loss:8.11
0_demographic_parity_difference:nan
0_equalized_odds_difference:nan
0_group_mean_prediction_difference:nan
1_accuracy:0.888
1_precision:0.9032
1_recall:0.888
1_f1:0.8944
1_roc_auc:0.762
1_log_loss:4.038
1_demographic_p

## 4. sklearn Fairness Workflow on Defended Data

This section trains a sklearn model on the Fairlearn-defended Adult dataset from Section 3 and reports both utility and group-level accuracy gap metrics.

In [4]:
sk_fair_model = FairlearnModelConfig(
    model_type="sklearn.linear_model.LogisticRegression",
    classifier=True,
    model_params={"max_iter": 200},
    data=fair_data,
)

sk_fair_scores = sk_fair_model(fair_data, model_file=sklearn_model_file.as_posix(), score_file=sklearn_score_file.as_posix(), )
sk_fair_scores

[DIAGNOSE] FairlearnScoreDictConfig.__call__: type(sensitive_features)=<class 'NoneType'>, sensitive_features=None
[DIAGNOSE] sensitive_features is None or length mismatch: type=<class 'NoneType'>, value=None, y_true type=<class 'pandas.core.series.Series'>, y_true len=9769, sensitive_features len=N/A


{'data_load_time': 0.0784239999999982,
 'data_sample_time': 0.0010200000000004,
 'pipeline_fit_time': 0.0361299999999999,
 'pipeline_fit_n': 39073.0,
 'pipeline_transform_time': 0.0054149999999992,
 'pipeline_transform_n': 9769.0,
 'num_classes': 2.0,
 'class_count_min': 11687.0,
 'class_count_max': 37155.0,
 'class_imbalance_ratio': 3.179173440574998,
 'mutual_information_mean': 0.0084220326712962,
 'mutual_information_max': 0.1113041974251816,
 'accuracy': 0.8127,
 'precision': 0.7975,
 'recall': 0.8127,
 'f1': 0.7979,
 'roc_auc': 0.7821,
 'log_loss': 0.4387,
 'demographic_parity_difference': 0.0311,
 'equalized_odds_difference': 0.2045,
 'group_mean_prediction_difference': 0.0311,
 '0_accuracy': 0.775,
 '0_precision': 0.7676,
 '0_recall': 0.775,
 '0_f1': 0.752,
 '0_roc_auc': 0.6678,
 '0_log_loss': 8.11,
 '0_demographic_parity_difference': nan,
 '0_equalized_odds_difference': nan,
 '0_group_mean_prediction_difference': nan,
 '1_accuracy': 0.888,
 '1_precision': 0.9032,
 '1_recall': 0

## 5. PyTorch Fairness Workflow

This section reuses the Fairlearn-preprocessed Adult dataset from Section 3 and fits a PyTorch model with fairness-aware group scoring. The Fairlearn defense here is the same `CorrelationRemover` preprocessing step, now evaluated through a torch model workflow.

In [5]:
from deckard.frameworks.pytorch import FairlearnPytorchDataConfig
from torch.utils.data import DataLoader

# Use the same data config as sklearn, but for torch
torch_fair_data = FairlearnPytorchDataConfig(dataset_name="deckard.frameworks.pytorch.fairness_data.TinyFairness", sensitive_columns=["_sensitive"])
torch_fair_data(data_file=(ARTIFACT_DIR / "torch_fair_data.pkl").as_posix())

# Debug: print type and structure of X_train
X_train = torch_fair_data.X_train
print("X_train type:", type(X_train))
in_features = None

# Robustly extract a sample from DataLoader or dataset
def get_in_features_from_loader_or_dataset(X_train):
    # If X_train is a DataLoader, get a batch
    if isinstance(X_train, DataLoader):
        try:
            batch = next(iter(X_train))
            # batch can be (inputs, labels) or just inputs
            if isinstance(batch, (tuple, list)) and len(batch) > 0:
                sample = batch[0]
                # If batch size > 1, take the first sample in the batch
                if hasattr(sample, 'shape') and sample.shape[0] > 1:
                    sample = sample[0]
            else:
                sample = batch
            # Now sample should be a tensor or array
            if hasattr(sample, 'shape') and len(sample.shape) > 0:
                return sample.shape[-1]
            elif hasattr(sample, 'size') and callable(sample.size):
                return sample.size(-1)
        except Exception as e:
            print("Could not extract in_features from DataLoader due to:", e)
    # If X_train is a dataset, try to get a sample
    elif hasattr(X_train, '__getitem__'):
        try:
            sample = X_train[0][0] if isinstance(X_train[0], (tuple, list)) else X_train[0]
            if hasattr(sample, 'shape') and len(sample.shape) > 0:
                return sample.shape[-1]
            elif hasattr(sample, 'size') and callable(sample.size):
                return sample.size(-1)
        except Exception as e:
            print("Could not extract in_features from dataset due to:", e)
    return None

in_features = get_in_features_from_loader_or_dataset(X_train)
print("Determined in_features:", in_features)
assert in_features is not None, "Could not determine input feature dimension for torch dataset."

torch_fair_model = FairlearnPytorchModelConfig(
    model_type="torch.nn.Linear",
    model_params={"in_features": int(in_features), "out_features": 2},
    classifier=True,
    criterion="CrossEntropyLoss",
    optimizer={"name": "torch.optim.Adam", "lr": 1e-3},
    fit_params={"nb_epochs": 2, "batch_size": 128},
    data=torch_fair_data,
    device="cpu",
)

torch_fair_scores = torch_fair_model(torch_fair_data, model_file=torch_model_file.as_posix(), score_file=torch_score_file.as_posix())

interesting_torch = [
    k for k in sorted(torch_fair_scores.keys())
    if "accuracy" in k or "sensitive_feature" in k or "f1" in k
]
print("torch fairness score keys:", interesting_torch[:20])
print({k: torch_fair_scores[k] for k in interesting_torch[:8] if isinstance(torch_fair_scores[k], (int, float))})


ModuleNotFoundError: No module named 'deckard.frameworks.utils'

## 6. Torch Fairness Experiment: Before/After Defense Comparison and Plots

This section uses `TorchExperimentConfig` to run before/after defense experiments with TinyNet and TinyFairness, then visualizes group accuracy improvements using seaborn and ROC AUC using Yellowbrick for both sklearn and torch models.

In [ ]:
from deckard.frameworks.pytorch.fairness_data import FairlearnPytorchDataConfig
from deckard.plugins.fairlearn.model import FairlearnPytorchModelConfig
import pickle

# Prepare data config for TorchExperimentConfig (before defense)
data_cfg = FairlearnPytorchDataConfig(
    dataset_name="deckard.data.fairness_pytorch.TinyFairness",
    sensitive_columns=["_sensitive"],
    classifier=True,
    train_size=0.8,
    test_size=0.2,
    random_state=42,
    stratify=True,
    device="cpu",
    data_params={"batch_size": 32, "num_workers": 0, "pin_memory": False},
)

# Model config (TinyNet, no defense)
in_features = get_in_features_from_loader_or_dataset(X_train)
torch_model_cfg = FairlearnPytorchModelConfig(
    model_type=TinyNet,
    classifier=True,
    model_params={"input_dim": int(in_features), "hidden_dim": 16, "output_dim": 2},
    fit_params={"nb_epochs": 2, "batch_size": 32},
    criterion="CrossEntropyLoss",
    optimizer={"name": "torch.optim.Adam", "lr": 1e-3},
    device="cpu",
)

# Run baseline experiment (no defense)
exp_before = TorchExperimentConfig(data=data_cfg, model=torch_model_cfg)
scores_before = exp_before()

# Prepare data config for TorchExperimentConfig (with defense)
data_cfg_def = FairlearnPytorchDataConfig(
    dataset_name="deckard.data.fairness_pytorch.TinyFairness",
    sensitive_columns=["_sensitive"],
    classifier=True,
    train_size=0.8,
    test_size=0.2,
    random_state=42,
    stratify=True,
    device="cpu",
    data_params={"batch_size": 32, "num_workers": 0, "pin_memory": False},
    fairness_defense={
        "step_name": "fairness_correlation_remover",
        "name": "fairlearn.preprocessing.CorrelationRemover",
        "alpha": 1.0,
    },
)

# Run defended experiment
exp_after = TorchExperimentConfig(data=data_cfg_def, model=torch_model_cfg)
scores_after = exp_after()

print("Torch baseline (no defense) scores:", scores_before)
print("Torch with defense scores:", scores_after)

# Seaborn comparison plot (accuracy/group accuracy)
def extract_group_accuracies(scores, prefix="group_"):
    return {k: v for k, v in scores.items() if k.startswith(prefix)}

group_acc_before = extract_group_accuracies(scores_before)
group_acc_after = extract_group_accuracies(scores_after)

# Yellowbrick plots are optional here; skip instead of failing the full notebook run.
for name, exp in (("before", exp_before), ("after", exp_after)):
    try:
        yb_cfg = YellowbrickPlotConfig(
            experiment=exp,
            plot_type="classification_report",
        )
        yb_cfg()
        print(f"Yellowbrick classification_report ({name}) rendered.")
    except Exception as exc:
        print(f"Yellowbrick classification_report ({name}) skipped: {exc}")

# DVC stage expects this artifact path; create a placeholder if no attack step produced one.
if not attack_file.exists():
    attack_file.parent.mkdir(parents=True, exist_ok=True)
    with attack_file.open("wb") as f:
        pickle.dump({"status": "placeholder", "reason": "no attack run in torch fairness section"}, f)
    print(f"Created placeholder attack artifact: {attack_file}")